In [5]:
import kagglehub
import pandas as pd
import numpy as np
import statsmodels.api as sm
import statsmodels.formula.api as smf
from linearmodels import PanelOLS, RandomEffects
import matplotlib.pyplot as plt
import japanize_matplotlib
import seaborn as sns
from stargazer.stargazer import Stargazer
from scipy.stats import norm
from scipy.stats import chi2
import os

pd.options.display.float_format = '{:.2f}'.format

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold
from sklearn.linear_model import LassoCV
from sklearn.preprocessing import StandardScaler
from linearmodels.iv import IV2SLS

from econml.dml import LinearDML

**discountのCATE**
median_playtime_forever

炎上の影響を高評価率によって引きたい

CCUではなく、他の妥当な比説明変数を使いたいが、量（Review）と質（プレイ時間中央値）の両方の関係も見たい

それらを踏まえた上で割引のCATEを測りたい

In [6]:
path = kagglehub.dataset_download("artermiloff/steam-games-dataset")

print("Path:", path)

csv_path = os.path.join(path, "games_march2025_cleaned.csv")

df = pd.read_csv(csv_path, index_col=False, quotechar='"')
df = df.drop(columns=['score_rank'])

pd.set_option('display.max_columns', None)

# pd.reset_option('display.max_columns')

df.head()

Path: /Users/kavuk/.cache/kagglehub/datasets/artermiloff/steam-games-dataset/versions/2


,appid,name,release_date,required_age,price,dlc_count,detailed_description,about_the_game,short_description,reviews,header_image,website,support_url,support_email,windows,mac,linux,metacritic_score,metacritic_url,achievements,recommendations,notes,supported_languages,full_audio_languages,packages,developers,publishers,categories,genres,screenshots,movies,user_score,positive,negative,estimated_owners,average_playtime_forever,average_playtime_2weeks,median_playtime_forever,median_playtime_2weeks,discount,peak_ccu,tags,pct_pos_total,num_reviews_total,pct_pos_recent,num_reviews_recent
0,730,Counter-Strike 2,2012-08-21,0,0.00,1,"For over two decades, Counter-Strike has offer...","For over two decades, Counter-Strike has offer...","For over two decades, Counter-Strike has offer...",NaN,https://shared.akamai.steamstatic.com/store_it...,http://counter-strike.net/,NaN,NaN,True,False,True,0,NaN,1,4401572,Includes intense violence and blood.,"['Czech', 'Danish', 'Dutch', 'English', 'Finni...","['English', 'Indonesian']","[{'title': 'Buy Counter-Strike 2', 'descriptio...",['Valve'],['Valve'],"['Multi-player', 'Cross-Platform Multiplayer',...","['Action', 'Free To Play']",['https://shared.akamai.steamstatic.com/store_...,['http://video.akamai.steamstatic.com/store_tr...,0,7480813,1135108,100000000 - 200000000,33189,879,5174,350,0,1212356,"{'FPS': 90857, 'Shooter': 65397, 'Multiplayer'...",86,8632939,82,96473
1,578080,PUBG: BATTLEGROUNDS,2017-12-21,0,0.00,0,"LAND, LOOT, SURVIVE! Play PUBG: BATTLEGROUNDS ...","LAND, LOOT, SURVIVE! Play PUBG: BATTLEGROUNDS ...",Play PUBG: BATTLEGROUNDS for free. Land on str...,NaN,https://shared.akamai.steamstatic.com/store_it...,https://www.pubg.com,https://support.pubg.com/hc/en-us,NaN,True,False,False,0,NaN,37,1732007,NaN,"['English', 'Korean', 'Simplified Chinese', 'F...",[],[],['PUBG Corporation'],"['KRAFTON, Inc.']","['Multi-player', 'PvP', 'Online PvP', 'Stats',...","['Action', 'Adventure', 'Massively Multiplayer...",['https://shared.akamai.steamstatic.com/store_...,[],0,1487960,1024436,50000000 - 100000000,0,0,0,0,0,616738,"{'Survival': 14838, 'Shooter': 12727, 'Battle ...",59,2513842,68,16720
2,570,Dota 2,2013-07-09,0,0.00,2,"The most-played game on Steam. Every day, mill...","The most-played game on Steam. Every day, mill...","Every day, millions of players worldwide enter...",“A modern multiplayer masterpiece.” 9.5/10 – D...,https://shared.akamai.steamstatic.com/store_it...,http://www.dota2.com/,NaN,NaN,True,True,True,90,https://www.metacritic.com/game/pc/dota-2?ftag...,0,14337,"Dota 2 includes fantasy violence, use of alcoh...","['Bulgarian', 'Czech', 'Danish', 'Dutch', 'Eng...","['English', 'Korean', 'Simplified Chinese', 'V...","[{'title': 'Buy Dota 2', 'description': '', 's...",['Valve'],['Valve'],"['Multi-player', 'Co-op', 'Steam Trading Cards...","['Action', 'Strategy', 'Free To Play']",['https://shared.akamai.steamstatic.com/store_...,['http://video.akamai.steamstatic.com/store_tr...,0,1998462,451338,200000000 - 500000000,43031,1536,898,892,0,555977,"{'Free to Play': 59933, 'MOBA': 20158, 'Multip...",81,2452595,80,29366
3,271590,Grand Theft Auto V Legacy,2015-04-13,17,0.00,0,"When a young street hustler, a retired bank ro...","When a young street hustler, a retired bank ro...",Grand Theft Auto V for PC offers players the o...,NaN,https://shared.akamai.steamstatic.com/store_it...,https://www.rockstargames.com/V/,https://support.rockstargames.com,NaN,True,False,False,96,https://www.metacritic.com/game/pc/grand-theft...,77,1803063,NaN,"['English', 'French', 'Italian', 'German', 'Sp...","['English', 'Spanish - Latin America']","[{'title': 'Buy Shark Cash Cards', 'descriptio...",['Rockstar North'],['Rockstar Games'],"['Single-player', 'Multi-player', 'PvP', 'Onli...","['Action', 'Adventure']",['https://shared.akamai.steamstatic.com/store_...,['http://video.akamai.steamstatic.com/store_tr...,0,1719950,250012,50000000 - 100000000,19323,771,7101,74,0,117698,"{'Open World': 32644, 'Action': 23539, 'M

In [ ]:
df['log_ccu'] = np.log(df['peak_ccu'] + 1)
df['log_positive'] = np.log(df['positive'] + 1)
df['log_negative'] = np.log(df['negative'] + 1)
df['log_median_playtime'] = np.log1p(df['median_playtime_forever'])
df['log_num_reviews_total'] = np.log1p(df['num_reviews_total'])

df['pub_id'] = pd.factorize(df['publishers'])[0]
df['main_genre'] = df['genres'].str.split(',').str[0]

# 1. カテゴリカル変数の数値化 (One-Hot Encoding)
# genres や categories は文字列のリスト（例: "Action;Indie"）なので、これを展開する
df_encoded = df.copy()

# ジャンルやカテゴリをダミー変数化（データ量が多い場合は上位10〜20個に絞る）
genres_dummies = df['genres'].str.get_dummies(sep=';').add_prefix('genre_')
categories_dummies = df['categories'].str.get_dummies(sep=';').add_prefix('cat_')


# 文字列のリスト（もしくはカンマ区切り文字列）をバラバラにして、上位20個の「単語」を抽出
# sep=';' か sep=',' かは、あなたのデータの実際の区切り文字に合わせてください
genre_counts = df['genres'].str.get_dummies(sep=',').sum()
top_20_names = genre_counts.sort_values(ascending=False).head(20).index

# これで「Action」単体の列、「Adventure」単体の列... が20個だけ作られる
genres_dummies = df['genres'].str.get_dummies(sep=',')[top_20_names].add_prefix('genre_')


# もし language_count がないなら作成
df_encoded['language_count'] = df['supported_languages'].str.split(',').str.len()

# 2. 統合された共変量 W の作成
# 数値変数 + カテゴリフラグ + IMR
W_cols = ['pct_pos_total', 'price', 'dlc_count', 'achievements', 'imr'] 
W_matrix = pd.concat([df_encoded[W_cols], genres_dummies, categories_dummies], axis=1)

# 3. 欠損値処理とデータの絞り込み
# DMLは欠損値を許容しないため、Wに含める行はすべて有効である必要がある
df_active = df_encoded[df_encoded['is_active'] == 1].copy()
# ... ここで W_matrix も df_active に合わせてスライス ...

# --- 前処理：フラグの作成 ---
df['is_early_access'] = df['genres'].fillna('').str.contains('Early Access').astype(int)
df['language_count'] = df['supported_languages'].fillna('').apply(lambda x: len(str(x).split(',')))

# --- 1. Selection Stage (Z の修正) ---
# カラムリストにある確実な変数を使用
Z_cols = ['language_count', 'windows', 'mac', 'linux', 'is_early_access', 'dlc_count']
Z = sm.add_constant(df[Z_cols])
selection_model = sm.Probit(df['is_active'], Z).fit()
# ... IMRの計算 ...

# --- 2. Outcome Stage (W の修正) ---
# ここは DML の「掃除」担当。genresをOne-hot展開してぶち込むのが理想
genres_dummies = df['genres'].str.get_dummies(sep=';').add_prefix('genre_')
df_final = pd.concat([df, genres_dummies], axis=1)

# 共変量 W に IMR とジャンルを統合
W_cols = ['price', 'dlc_count', 'imr', 'pct_pos_total'] + list(genres_dummies.columns)
# ※ df_active に対して W を作成

df.sample(n=10)

/opt/anaconda3/lib/python3.13/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: divide by zero encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)


KeyboardInterrupt: 

In [ ]:
# --- 0. 関数定義：ジャンル・カテゴリの多重展開 ---
def get_top_dummies(df, column, sep=';', top_n=20):
    # 分割してダミー化
    dummies = df[column].fillna('').str.get_dummies(sep=sep)
    # 頻度上位のみ抽出
    top_cols = dummies.sum().sort_values(ascending=False).head(top_n).index
    return dummies[top_cols].add_prefix(f"{column.rstrip('s')}_")

# --- 1. Selection Stage (Heckman第一段階) ---
# 前処理
df['is_early_access'] = df['genres'].fillna('').str.contains('Early Access').astype(int)
df['language_count'] = df['supported_languages'].fillna('').str.split(',').str.len()

# Z（選択式に含まれるが、売上には直接影響しにくい「操作変数」的変数を含む）
Z_cols = ['language_count', 'windows', 'mac', 'linux', 'is_early_access', 'dlc_count']
Z = sm.add_constant(df[Z_cols])

# Probit推定
selection_model = sm.Probit(df['is_active'], Z).fit()

# IMR (逆ミルズ比) の計算
prob = selection_model.predict(Z)
phi = norm.pdf(norm.ppf(np.clip(prob, 0.001, 0.999)))
df['imr'] = phi / np.clip(prob, 0.001, 0.999)

# --- 2. Outcome Stage 用のデータ整理 ---
# ジャンル(20個)とカテゴリ(15個)を展開
genres_dummies = get_top_dummies(df, 'genres', sep=';', top_n=20)
categories_dummies = get_top_dummies(df, 'categories', sep=';', top_n=15)

# 分析対象の絞り込み（アクティブかつ、割引情報があるもの）
df_active = df[df['is_active'] == 1].copy()

# 共変量 W の作成 (IMR + 数値変数 + ジャンル + カテゴリ)
W_numeric_cols = ['price', 'dlc_count', 'imr', 'pct_pos_total']
W_matrix = pd.concat([
    df_active[W_numeric_cols],
    genres_dummies.loc[df_active.index],
    categories_dummies.loc[df_active.index]
], axis=1).fillna(0)

# DMLに渡す変数の確定
Y = np.log1p(df_active['peak_ccu']) # Outcome: 例としてCCUの対数
T = df_active['discount_pct']      # Treatment: 割引率
X = df_active[['metacritic']]      # Heterogeneity: メタスコアによる効果の違いを見る
W = W_matrix.values               # Controls: 「掃除」用変数群

# --- 3. DML の実行 ---
# LassoCV を使うことで、40近い共変量の中から重要なものだけを自動選択させる
from econml.dml import LinearDML

est = LinearDML(
    model_y=LassoCV(cv=3),
    model_t=LassoCV(cv=3),
    discrete_treatment=False
)

print(f"Final W shape: {W.shape} (Variables: {W_matrix.columns.tolist()})")
est.fit(Y, T, X=X, W=W)

In [ ]:
# 1. データの準備（Sand_03 を想定）
# T: 処置（割引率 0-100）
# X: 条件（CATEを見たい変数：メタスコアなど）
# W: 共変量（掃除したい変数：ポジティブ率、価格、タグ、ジャンルなど）
# Y_vol: 量（レビュー数） / Y_qual: 質（中央値プレイ時間）

T = df['discount'].values
X = df[['metacritic_score']].values  # メタスコアによって効果が変わるか見たい
W = df[['pct_pos_total', 'price', 'dlc_count', 'is_early_access']].values # 炎上率等で掃除

# 目的変数は対数変換（0を避けるため +1）
Y_vol = np.log(df['num_reviews_total'] + 1).values
Y_qual = np.log(df['median_playtime_forever'] + 1).values

# 2. モデルの定義（Double Machine Learning）
# 機械学習（Random Forest）で T と Y のノイズを掃除する
def estimate_cate(Y, T, X, W):
    est = LinearDML(
        model_y=RandomForestRegressor(n_estimators=100, max_depth=5, min_samples_leaf=10),
        model_t=RandomForestRegressor(n_estimators=100, max_depth=5, min_samples_leaf=10),
        discrete_treatment=False
    )
    est.fit(Y, T, X=X, W=W)
    return est

# 3. 「量」と「質」のそれぞれで CATE を推定
print("Estimating Volume Effect (Reviews)...")
est_vol = estimate_cate(Y_vol, T, X, W)

print("Estimating Quality Effect (Playtime)...")
est_qual = estimate_cate(Y_qual, T, X, W)

# 4. 可視化：メタスコアが上がると、割引の「量」と「質」への効果はどう変わるか
test_X = np.linspace(X.min(), X.max(), 100).reshape(-1, 1)
cate_vol = est_vol.effect(test_X)
cate_qual = est_qual.effect(test_X)

plt.figure(figsize=(10, 6))
plt.plot(test_X, cate_vol, label='CATE: Volume (Reviews)', color='blue')
plt.plot(test_X, cate_qual, label='CATE: Quality (Playtime)', color='green')
plt.axhline(0, color='red', linestyle='--')
plt.xlabel('Metacritic Score')
plt.ylabel('Marginal Treatment Effect of Discount')
plt.title('Quantity vs. Quality Trade-off in Discounting')
plt.legend()
plt.show()

KeyError: "['is_early_access'] not in index"

でもこれ同時方程式じゃ。IVーDML使った方がいいんじゃないか。

いや売れてるゲームほど割引するって仮説は繊細すぎるか、一旦この2変数とdiscountの相関見るか。だって売れてないゲームも割引するだろうから打ち消されそうだ

でもそもそもマジで一応市場に出ただけみたいなクソゲーもdiscountしないだろうしやっぱりreviewをheckmanで切りたくなってしまうな。難しい

Low-end (クソゲー): そもそも割引すら設定されず放置される。

Mid-range (普通): 売れないから割引する（負の相関）。

High-end (人気作): セールイベントで戦略的に割引し、さらに売れる（正の相関）。

これらがデータセット内で混ざると、全体の係数は「0」に近づき、「割引は意味がない」という誤った結論を導きます。

4. 暫定的な解決策：Heckman-DML 的なアプローチ

最新の論文でも議論される内容ですが、現実的な解として以下を提案します。

Step 1: フィルタリング（Heckman の精神）
「レビューが 10 以上」などの最低限の足切りを行い、さらに「過去に一度でも割引をしたことがあるか」というプロビットを回して IMR（逆ミルズ比） を出す。

Step 2: DML への投入
その IMR をコントロール変数として DML にぶち込みます。これで「割引戦略をとるようなゲーム層」における、純粋な割引効果を推定できます。

In [ ]:
# 1. Selection Stage: 逆ミルズ比 (IMR) の算出
# 全データを使用して「一定以上のレビューがあるか」という生存フラグを作成
df['is_observed'] = (df['num_reviews_total'] > 10).astype(int)

# 選択方程式の変数 Z (排外制約: 生存には効くが、売上そのものには直接関わらないもの)
# 例: 発売からの日数、言語数、特定のカテゴリフラグなど
Z_cols = ['language_count', 'is_early_access', 'windows', 'mac', 'linux']
Z = sm.add_constant(df[Z_cols])

# プロビット回帰の実行
probit_model = sm.Probit(df['is_observed'], Z).fit()

# 各サンプルに対して逆ミルズ比 (IMR) を計算
# phi(z)/Phi(z)
z_score = probit_model.predict(Z)
# predictは確率を出すので、逆関数で標準正規分布のzスコアに戻す必要がある
z_val = norm.ppf(np.clip(z_score, 0.001, 0.999))
df['imr'] = norm.pdf(z_val) / norm.cdf(z_val)

# 2. DML Stage: 選択バイアスを補正した CATE 推定
# 分析対象を「観測されたデータ」に絞る
df_sub = df[df['is_observed'] == 1].copy().dropna(subset=['metacritic_score', 'discount'])

# Y: 目的変数 (量: reviews / 質: playtime)
Y = np.log(df_sub['num_reviews_total'] + 1).values

# T: 処置変数 (割引率)
T = df_sub['discount'].values

# X: 条件変数 (CATEを見たい軸)
X = df_sub[['metacritic_score']].values

# W: 共変量 (IMRをここに追加するのがキモ)
# 他にコントロールしたい変数（炎上率など）も入れる
W = df_sub[['pct_pos_total', 'price', 'imr']].values

# LinearDML の定義
est = LinearDML(
    model_y=RandomForestRegressor(n_estimators=100, max_depth=5),
    model_t=RandomForestRegressor(n_estimators=100, max_depth=5),
    discrete_treatment=False
)

# 学習
print("Fitting Heckman-DML model...")
est.fit(Y, T, X=X, W=W)

# 3. 結果の解釈
# メタスコアごとの割引の限界効果を表示
test_X = np.linspace(X.min(), X.max(), 100).reshape(-1, 1)
treatment_effects = est.effect(test_X)

import matplotlib.pyplot as plt
plt.plot(test_X, treatment_effects)
plt.axhline(0, color='r', linestyle='--')
plt.title("CATE of Discount (Heckman-DML Corrected)")
plt.xlabel("Metacritic Score")
plt.ylabel("Treatment Effect on Log Reviews")
plt.show()

でもこれヘックマンの必然性あるのか？対数変換しても尖度がこの程度あるから切った。みたいなのって妥当性あるのか？Truncated Regressionの方がいいか？でも、そもそも売れてなくて、戦略としてすら割引をしないゲームを第一段階で省いて、戦略として割引をすることが意味を持ちうるライン（reviewが正）で分けて分析するっていうのは十分な根拠か


In [ ]:
# 1. Selection Stage (Heckman): 戦略的参入のモデリング
# 「レビューが1つ以上あるか」を市場参入の証拠とする
df['is_active'] = (df['num_reviews_total'] > 0).astype(int)

# 選択方程式の共変量 Z (排外制約: 露出やプラットフォーム対応など)
Z_cols = ['language_count', 'windows', 'mac', 'linux', 'is_early_access']
Z = sm.add_constant(df[Z_cols])

# プロビットモデルで「参入確率」を推定
selection_model = sm.Probit(df['is_active'], Z).fit()

# 逆ミルズ比 (IMR) の算出
# 確率 P から標準正規分布の z 値を逆算し、IMRを導出
prob = selection_model.predict(Z)
z_val = norm.ppf(np.clip(prob, 0.0001, 0.9999))
df['imr'] = norm.pdf(z_val) / norm.cdf(z_val)

# 2. Outcome Stage (DML): 参入個体における割引の CATE 推定
# 「市場に参入している」個体のみを抽出
df_active = df[df['is_active'] == 1].copy().dropna(subset=['metacritic_score', 'discount'])

# Y: 目的変数 (量: reviews / 質: playtime)
# 対数変換で分布を整えつつ、IMRで選択バイアスを制御する
Y = np.log(df_active['num_reviews_total'] + 1).values
T = df_active['discount'].values
X = df_active[['metacritic_score']].values  # メタスコアで効果の差（CATE）を見る
W = df_active_W_matrix.values

# DML モデルの定義
est = LinearDML(
    model_y=RandomForestRegressor(n_estimators=100, max_depth=5, min_samples_leaf=10),
    model_t=RandomForestRegressor(n_estimators=100, max_depth=5, min_samples_leaf=10),
    discrete_treatment=False
)

print("Running Heckman-DML for CATE estimation...")
est.fit(Y, T, X=X, W=W)

# 3. 可視化：メタスコアによる割引効果の変遷
test_X = np.sort(X, axis=0)
cate_effects = est.effect(test_X)
lb, ub = est.effect_interval(test_X, alpha=0.05) # 95%信頼区間

plt.figure(figsize=(10, 6))
plt.fill_between(test_X.flatten(), lb, ub, color='gray', alpha=0.2, label='95% CI')
plt.plot(test_X, cate_effects, label='Marginal Effect of Discount', color='blue', lw=2)
plt.axhline(0, color='red', linestyle='--')
plt.title("Heterogeneous Treatment Effect of Discounting\n(Controlled for Selection Bias via IMR)")
plt.xlabel("Metacritic Score")
plt.ylabel("Effect on Log(Total Reviews)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()